Leiden implementation

In [1]:
import sys
print(sys.executable)

/Users/annafabiagai/miniconda3/envs/lfr-env/bin/python


In [2]:
%pip install cdlib python-igraph leidenalg networkx


  Using cached python_igraph-1.0.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached plotly-6.5.0-py3-none-any.whl.metadata (8.5 kB)
Using cached python_igraph-1.0.0-py3-none-any.whl (9.2 kB)
Using cached plotly-6.5.0-py3-none-any.whl (9.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [plotly]2m2/3 [plotly]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import cdlib
import igraph
import leidenalg

print("cdlib", cdlib.__version__)
print("igraph", igraph.__version__)


cdlib 0.4.0
igraph 1.0.0


**PARTE PROGETTO EFFETTIVO**





N= 500       mu=0.1  

20 test senza seed, NMI , F-SCORE, ARI as average of the all scores 

In [11]:
import numpy as np
import pandas as pd
import networkx as nx

from cdlib.benchmark import LFR
from cdlib import algorithms, evaluation


def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    # assicurati stesso insieme di nodi
    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0

    # conteggio su tutte le coppie (i<j)
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif (not same_pred) and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1

# ---------- Parametri fissi LFR ----------
LFR_PARAMS = dict(
    n=500,
    tau1=3,
    tau2=2.0,
    mu=0.1,
    average_degree=5,
    min_community=20,
    tol=1e-5,
    max_iters=1000,
)

N_TESTS = 20
MAX_ATTEMPTS = 800   # alza un po' per stare tranquilla

BASE_SEED = None     # None = casuale vero; metti un int per riproducibilità
rng = np.random.default_rng(BASE_SEED)

rows = []
attempts = 0
test_id = 0
skipped = 0

while test_id < N_TESTS and attempts < MAX_ATTEMPTS:
    attempts += 1
    seed = int(rng.integers(0, 2**31 - 1))

    try:
        G, coms = LFR(**LFR_PARAMS, seed=seed)
    except nx.ExceededMaxIterations:
        skipped += 1
        continue

    leiden_coms = algorithms.leiden(G)

    # Metriche (NMI, ARI con cdlib)
    nmi = evaluation.normalized_mutual_information(leiden_coms, coms).score
    ari = evaluation.adjusted_rand_index(leiden_coms, coms).score

    # F-score/F1 pairwise calcolato a mano (robusto)
    f1 = pairwise_f1(leiden_coms.communities, coms.communities)

    test_id += 1
    rows.append(
        {"test": test_id, "seed": seed, "NMI": nmi, "F_score": f1, "ARI": ari}
    )

if test_id < N_TESTS:
    raise RuntimeError(
        f"Riusciti solo {test_id}/{N_TESTS} test validi dopo {attempts} tentativi. "
        f"Seed saltati: {skipped}. Aumenta MAX_ATTEMPTS o rilassa i parametri LFR."
    )

df = pd.DataFrame(rows)

mean_row = {
    "test": "MEAN",
    "seed": "",
    "NMI": df["NMI"].mean(),
    "F_score": df["F_score"].mean(),
    "ARI": df["ARI"].mean(),
}

df_out = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

print(f"Tentativi totali: {attempts} | Seed saltati: {skipped}")
print(df_out)

out_path = "leiden_lfr_1st_tests_results.csv"
df_out.to_csv(out_path, index=False)
print(f"\nSalvato in: {out_path}")


Tentativi totali: 31 | Seed saltati: 11
    test        seed       NMI   F_score       ARI
0      1   983690662  0.989108  0.983736  0.982954
1      2  1345483375  0.981442  0.974834  0.973456
2      3  1677714555  0.963840  0.968406  0.963801
3      4   689634131  0.998581  0.998517  0.998428
4      5   993959332  0.985697  0.975743  0.974741
5      6   849373119  0.976627  0.960391  0.958589
6      7  2066018646  0.995953  0.993593  0.993300
7      8  2114673881  0.994824  0.991890  0.991539
8      9  1809877132  0.982020  0.979318  0.977804
9     10   780254200  0.985635  0.977130  0.975636
10    11   375937035  0.992367  0.993115  0.992359
11    12  2112916873  0.990520  0.982875  0.982130
12    13  1245634399  0.993015  0.989989  0.989480
13    14  1605887485  0.990195  0.985981  0.985264
14    15  1404630998  0.995203  0.993738  0.993364
15    16  1470329478  0.990519  0.990830  0.990122
16    17   159025029  0.985468  0.978221  0.977224
17    18  2018007925  0.989971  0.984196  

**SECONDO TEST**

N=1000 mu=0.1 tau=2
20 test ciascuno

In [10]:
import numpy as np
import pandas as pd
import networkx as nx

from cdlib.benchmark import LFR
from cdlib import algorithms, evaluation


def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    # assicurati stesso insieme di nodi
    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0

    # conteggio su tutte le coppie (i<j)
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif (not same_pred) and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1


LFR_PARAMS = dict(
    n=1000,
    tau1=3,
    tau2=2.0,
    mu=0.1,
    average_degree=5,
    min_community=20,
    tol=1e-5,
    max_iters=1000,
)

N_TESTS = 20
MAX_ATTEMPTS = 800   

BASE_SEED = None     
rng = np.random.default_rng(BASE_SEED)

rows = []
attempts = 0
test_id = 0
skipped = 0

while test_id < N_TESTS and attempts < MAX_ATTEMPTS:
    attempts += 1
    seed = int(rng.integers(0, 2**31 - 1))

    try:
        G, coms = LFR(**LFR_PARAMS, seed=seed)
    except nx.ExceededMaxIterations:
        skipped += 1
        continue

    leiden_coms = algorithms.leiden(G)

    # Metriche (NMI, ARI con cdlib)
    nmi = evaluation.normalized_mutual_information(leiden_coms, coms).score
    ari = evaluation.adjusted_rand_index(leiden_coms, coms).score

    # F-score/F1 pairwise calcolato a mano (robusto)
    f1 = pairwise_f1(leiden_coms.communities, coms.communities)

    test_id += 1
    rows.append(
        {"test": test_id, "seed": seed, "NMI": nmi, "F_score": f1, "ARI": ari}
    )

if test_id < N_TESTS:
    raise RuntimeError(
        f"Riusciti solo {test_id}/{N_TESTS} test validi dopo {attempts} tentativi. "
        f"Seed saltati: {skipped}. Aumenta MAX_ATTEMPTS o rilassa i parametri LFR."
    )

df = pd.DataFrame(rows)

mean_row = {
    "test": "MEAN",
    "seed": "",
    "NMI": df["NMI"].mean(),
    "F_score": df["F_score"].mean(),
    "ARI": df["ARI"].mean(),
}

df_out = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

print(f"Tentativi totali: {attempts} | Seed saltati: {skipped}")
print(df_out)

out_path = "leiden_lfr_2nd_test_results.csv"
df_out.to_csv(out_path, index=False)
print(f"\nSalvato in: {out_path}")


Tentativi totali: 31 | Seed saltati: 11
    test        seed       NMI   F_score       ARI
0      1  1349329054  0.982314  0.979700  0.978363
1      2    51903692  0.978528  0.950524  0.949020
2      3   503794755  0.990069  0.969903  0.969195
3      4    85976550  0.989183  0.991786  0.990295
4      5   392614447  0.974991  0.918085  0.916088
5      6   353267322  0.982533  0.975507  0.974140
6      7   181320989  0.977884  0.946795  0.945070
7      8  2098559850  0.987903  0.974513  0.973589
8      9  1157138803  0.980849  0.949499  0.948058
9     10  1437751767  0.986341  0.950026  0.948661
10    11    47501520  0.988120  0.976120  0.975519
11    12   866034973  0.984992  0.974275  0.972797
12    13    56154805  0.983047  0.978970  0.977955
13    14   691296902  0.989503  0.987121  0.986427
14    15  1452621495  0.993343  0.989183  0.988822
15    16  2043781390  0.994695  0.994025  0.993782
16    17   189661758  0.988042  0.981460  0.980808
17    18   665103423  0.986546  0.975870  

Third test N=1000 mu=0.3 

In [8]:
import numpy as np
import pandas as pd
import networkx as nx

from cdlib.benchmark import LFR
from cdlib import algorithms, evaluation


def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    # assicurati stesso insieme di nodi
    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0

    # conteggio su tutte le coppie (i<j)
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif (not same_pred) and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1

# ---------- Parametri fissi LFR ----------
LFR_PARAMS = dict(
    n=1000,
    tau1=3,
    tau2=2.0,
    mu=0.3,
    average_degree=5,
    min_community=20,
    tol=1e-5,
    max_iters=1000,
)

N_TESTS = 20
MAX_ATTEMPTS = 800   # alza un po' per stare tranquilla

BASE_SEED = None     # None = casuale vero; metti un int per riproducibilità
rng = np.random.default_rng(BASE_SEED)

rows = []
attempts = 0
test_id = 0
skipped = 0

while test_id < N_TESTS and attempts < MAX_ATTEMPTS:
    attempts += 1
    seed = int(rng.integers(0, 2**31 - 1))

    try:
        G, coms = LFR(**LFR_PARAMS, seed=seed)
    except nx.ExceededMaxIterations:
        skipped += 1
        continue

    leiden_coms = algorithms.leiden(G)

    # Metriche (NMI, ARI con cdlib)
    nmi = evaluation.normalized_mutual_information(leiden_coms, coms).score
    ari = evaluation.adjusted_rand_index(leiden_coms, coms).score

    # F-score/F1 pairwise calcolato a mano (robusto)
    f1 = pairwise_f1(leiden_coms.communities, coms.communities)

    test_id += 1
    rows.append(
        {"test": test_id, "seed": seed, "NMI": nmi, "F_score": f1, "ARI": ari}
    )

if test_id < N_TESTS:
    raise RuntimeError(
        f"Riusciti solo {test_id}/{N_TESTS} test validi dopo {attempts} tentativi. "
        f"Seed saltati: {skipped}. Aumenta MAX_ATTEMPTS o rilassa i parametri LFR."
    )

df = pd.DataFrame(rows)

mean_row = {
    "test": "MEAN",
    "seed": "",
    "NMI": df["NMI"].mean(),
    "F_score": df["F_score"].mean(),
    "ARI": df["ARI"].mean(),
}

df_out = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

print(f"Tentativi totali: {attempts} | Seed saltati: {skipped}")
print(df_out)

out_path = "leiden_lfr_3th_tests_results.csv"
df_out.to_csv(out_path, index=False)
print(f"\nSalvato in: {out_path}")


Tentativi totali: 20 | Seed saltati: 0
    test        seed       NMI   F_score       ARI
0      1  1015483653  0.426458  0.203588  0.173577
1      2  1666155612  0.416249  0.202487  0.171483
2      3    17995540  0.374078  0.172935  0.139798
3      4  1645916201  0.429808  0.246291  0.210807
4      5  1058115652  0.341584  0.239655  0.188770
5      6   767659173  0.388411  0.249535  0.214168
6      7  1017514842  0.396484  0.256605  0.218301
7      8   251523943  0.433533  0.259151  0.224716
8      9   746221409  0.413604  0.255094  0.218403
9     10    83853033  0.268069  0.231045  0.172373
10    11   607986143  0.400003  0.196860  0.166427
11    12   514542794  0.406625  0.206012  0.173844
12    13   275964401  0.314797  0.220340  0.161419
13    14  1251913274  0.416436  0.201078  0.171624
14    15  1398249522  0.309517  0.160391  0.118461
15    16  1621044496  0.372974  0.154606  0.123259
16    17  1244552567  0.472601  0.214758  0.189518
17    18   274417184  0.375478  0.268695  0

**UNICO CODICE CHE CICLA PER I VARI VALORI DI N E MU E SALVA TUTTO**

In [4]:
import numpy as np
import pandas as pd
import networkx as nx

from cdlib.benchmark import LFR
from cdlib import algorithms, evaluation


def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif not same_pred and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0


# ---------- Configurazioni grafi ----------
GRAPHS = {
    "Graph_1": {"n": 500,  "mu": 0.1},
    "Graph_2": {"n": 1000, "mu": 0.1},
    "Graph_3": {"n": 1000, "mu": 0.3},
}

BASE_LFR_PARAMS = dict(
    tau1=3,
    tau2=2.0,
    average_degree=5,
    min_community=20,
    tol=1e-5,
    max_iters=1000,
)

N_TESTS = 20
MAX_ATTEMPTS = 800
BASE_SEED = None
rng = np.random.default_rng(BASE_SEED)

rows = []

for graph_id, gparams in GRAPHS.items():
    test_id = 0
    attempts = 0
    skipped = 0

    while test_id < N_TESTS and attempts < MAX_ATTEMPTS:
        attempts += 1
        seed = int(rng.integers(0, 2**31 - 1))

        try:
            G, coms = LFR(
                **BASE_LFR_PARAMS,
                n=gparams["n"],
                mu=gparams["mu"],
                seed=seed,
            )
        except nx.ExceededMaxIterations:
            skipped += 1
            continue

        leiden_coms = algorithms.leiden(G)

        nmi = evaluation.normalized_mutual_information(leiden_coms, coms).score
        ari = evaluation.adjusted_rand_index(leiden_coms, coms).score
        f1  = pairwise_f1(leiden_coms.communities, coms.communities)

        test_id += 1
        rows.append({
            "graph": graph_id,
            "test": test_id,
            "seed": seed,
            "NMI": nmi,
            "F_score": f1,
            "ARI": ari
        })

    if test_id < N_TESTS:
        raise RuntimeError(
            f"{graph_id}: solo {test_id}/{N_TESTS} test validi "
            f"dopo {attempts} tentativi (skipped={skipped})"
        )

# ---------- DataFrame finale ----------
df = pd.DataFrame(rows)

# Media per grafo
mean_df = (
    df.groupby("graph")[["NMI", "F_score", "ARI"]]
      .mean()
      .reset_index()
)
mean_df["test"] = "MEAN"
mean_df["seed"] = ""

df_out = pd.concat([df, mean_df], ignore_index=True)

out_path = "leiden_lfr_all_graphs_results.csv"
df_out.to_csv(out_path, index=False)

print(df_out)
print(f"\nSalvato in: {out_path}")


Note: to be able to use all crisp methods, you need to install some additional packages:  {'infomap', 'bayanpy', 'wurlitzer', 'graph_tool'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'ASLPAw', 'pyclustering'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'infomap', 'wurlitzer'}
      graph  test        seed       NMI   F_score       ARI
0   Graph_1     1   993593565  0.991236  0.987751  0.987074
1   Graph_1     2  1944449831  0.988560  0.985769  0.984783
2   Graph_1     3  1523542751  0.993535  0.989992  0.989557
3   Graph_1     4   367859307  0.984654  0.982091  0.980724
4   Graph_1     5  1893809843  0.995715  0.994360  0.994051
..      ...   ...         ...       ...       ...       ...
58  Graph_3    19   891745166  0.351193  0.363766  0.308643
59  Graph_3    20   250568079  0.400958  0.256616  0.219268
60  Graph_1  MEAN              0.992099  0.987706  0.987019
61  Graph_2  MEAN           

**3 TEST CON GLI EMBEDDINGS**


In [5]:
import networkx as nx
import random
import math
import numpy as np

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from cdlib.benchmark import LFR
from node2vec import Node2Vec


def generate_original_graph(N, mu):
    while True:
        try:  # needed because sometimes LFR is not able to assign communities
            # generate LFR graph
            # TODO: VALUTARE UN ATTIMO COME SETTARE average_degree E min_community
            G, ground_truth_comms = LFR(n=N, mu=mu,
                                        tau1=3, tau2=2, min_community=N / 20,
                                        average_degree=random.randint(math.floor(N / 100),
                                                                      math.ceil(N / 50)),
                                        seed=random.randint(1, 10000)
                                        )
            # if successful remove self loops and return the graph + communities
            G.remove_edges_from(nx.selfloop_edges(G))
            return G, ground_truth_comms

        # if errors occur retry / give warning
        except nx.ExceededMaxIterations:
            print("Generation failed (ExceededMaxIterations). Retrying...")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            raise e

def get_reconstructed_graph(G, reconstruction, metric):

    # if input is not correct stop the execution
    if (reconstruction != "knn" and reconstruction != "threshold") or (metric != "cosine" and metric != "dot"):
        print("Incorrect reconstruction method or metric")
        return None

    # create node2vec model and get the embeddings
    # P E Q MOTIVATI DAL PAPER DI NODE2VEC (STESSI NUMERI DEL CASE STUDY), N=64 PERCHÈ USIAMO GRAFI PICCOLI (evita curse of dim)
    node2vec_model = Node2Vec(G, dimensions=64,
                              q=0.5, p=1,
                              seed=123)
    embeddings = node2vec_model.fit() # output works like a dictionary, key = node_id

    # convert embeddings into a numpy array
    model_wv = embeddings.wv  # this gets all keyed vectors in the attribute .wv
    nodes = sorted(list(G.nodes()))  # get nodes list from original graph
    X = np.array([model_wv[str(n)] for n in nodes])  # do the lookup in model_wv node by node,
                                                     # convert 'n' to str() because node2vec stores keys as strings
    # initialize empty graph and edge list
    reconstructed_graph = nx.Graph()
    reconstructed_graph.add_nodes_from(nodes)
    edge_list = []

    # based on the selected reconstruction method and metric build the graph
    if reconstruction == "knn":
        # get k for k-NN as the average node degree
        k = int(np.round((2 * G.number_of_edges()) / G.number_of_nodes()))

        if metric == "cosine":
            # get knn model
            knn = NearestNeighbors(n_neighbors=k + 1, # use k+1 because the first neighbor is always the node itself (distance=0)
                                   metric=metric)
            knn.fit(X)
            distances, indices = knn.kneighbors(X)

            # get the list of edges and weights to add to the reconstructed graph
            for i, (neighbor_indices, neighbor_dists) in enumerate(zip(indices, distances)):
                u = nodes[i]
                for neighbor_idx, dist in zip(neighbor_indices[1:], neighbor_dists[1:]):
                    v = nodes[neighbor_idx]
                    weight = 1 - dist # convert distance to similarity
                    edge_list.append((u, v, {'weight': weight}))

        elif metric == "dot":
            # compute similarity matrix and fill diagonal with -inf to avoid self loops
            sim_matrix = np.dot(X, X.T)
            np.fill_diagonal(sim_matrix, -np.inf)

            # for every node find the k highest values
            # argpartition puts the top k indices at the end of the array
            # take the last k indices
            top_k_indices = np.argpartition(sim_matrix, -k, axis=1)[:, -k:]

            # get the list of edges and weights to add to the reconstructed graph
            for i, neighbor_indices in enumerate(top_k_indices):
                u = nodes[i]
                for neighbor_idx in neighbor_indices:
                    v = nodes[neighbor_idx]
                    weight = sim_matrix[i, neighbor_idx] # get the dot product as weight
                    edge_list.append((u, v, {'weight': float(weight)}))

    else: # if reconstruction == "thresholding"
        t = 0.5  # set threshold
        if metric == "cosine":
            # compute cosine similarity matrix and remove self loops
            sim_matrix = cosine_similarity(X)
            np.fill_diagonal(sim_matrix, 0)

            # gets indices of node couples that overcome the threshold that are
            # located in the upper triangle of the matrix (avoids duplicates)
            rows, cols = np.where(np.triu(sim_matrix, k=1) >= t)

            # get the list of edges and weights to add to the reconstructed graph
            for u_idx, v_idx in zip(rows, cols):
                u, v = nodes[u_idx], nodes[v_idx]
                weight = sim_matrix[u_idx, v_idx]
                edge_list.append((u, v, {'weight': float(weight)}))

        elif metric == "dot":
            # compute similarity matrix and fill diagonal with -inf to avoid self loops
            sim_matrix = np.dot(X, X.T)
            np.fill_diagonal(sim_matrix, -np.inf)

            # do min-max normalization to bring range to [0, 1]
            valid_values = sim_matrix[sim_matrix != -np.inf] # ignore -inf values
            min_val = valid_values.min()
            max_val = valid_values.max()
            sim_matrix_norm = (sim_matrix - min_val) / (max_val - min_val)

            # apply threshold (same line of code of cosine)
            rows, cols = np.where(np.triu(sim_matrix_norm, k=1) >= t)

            # get the list of edges and weights to add to the reconstructed graph
            for u_idx, v_idx in zip(rows, cols):
                u, v = nodes[u_idx], nodes[v_idx]
                weight = sim_matrix_norm[u_idx, v_idx]
                edge_list.append((u, v, {'weight': float(weight)}))

    # add edges to the graph
    reconstructed_graph.add_edges_from(edge_list)

    return reconstructed_graph

In [39]:
import numpy as np
import pandas as pd
import networkx as nx

from cdlib.benchmark import LFR
from cdlib import algorithms, evaluation


N_TESTS = 20
rows = []

def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    # assicurati stesso insieme di nodi
    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0

    # conteggio su tutte le coppie (i<j)
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif (not same_pred) and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return f1

for test_id in range(1, N_TESTS + 1):

    G, ground_truth = generate_original_graph(N=1000, mu=0.3)

    Gr = get_reconstructed_graph(
        G,
        reconstruction="knn",           #knn / threshold
        metric="cosine"                    #dot / cosine
         )

    leiden_coms = algorithms.leiden(Gr)

    nmi = evaluation.normalized_mutual_information(leiden_coms, ground_truth).score
    ari = evaluation.adjusted_rand_index(leiden_coms, ground_truth).score
    f1  = pairwise_f1(leiden_coms.communities, ground_truth.communities)

    rows.append({
        "test": test_id,
        "NMI": nmi,
        "F_score": f1,
        "ARI": ari
    })

df = pd.DataFrame(rows)

mean_row = {
    "test": "MEAN",
    "NMI": df["NMI"].mean(),
    "F_score": df["F_score"].mean(),
    "ARI": df["ARI"].mean(),
}

df_out = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
print(df_out)


Generating walks (CPU: 1): 100%|██████████| 10/10 [00:05<00:00,  1.75it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Generating walks (CPU: 1): 100%|██████████| 10/10 [00:05<00:00,  1.96it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Generating walks (CPU: 1): 100%|██████████| 10/10 [00:05<00:00,  1.98it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.w

    test       NMI   F_score       ARI
0      1  0.809864  0.837906  0.799317
1      2  0.803645  0.825689  0.784969
2      3  0.812345  0.841460  0.803600
3      4  0.816223  0.845956  0.809150
4      5  0.799730  0.817391  0.775128
5      6  0.801063  0.826486  0.785323
6      7  0.806889  0.835005  0.795665
7      8  0.814378  0.845071  0.808347
8      9  0.801169  0.827711  0.786945
9     10  0.813320  0.842416  0.805006
10    11  0.821056  0.851526  0.815851
11    12  0.803972  0.831666  0.791710
12    13  0.808055  0.828027  0.787731
13    14  0.804074  0.831444  0.791764
14    15  0.813185  0.843433  0.806228
15    16  0.805394  0.826849  0.785668
16    17  0.820143  0.850006  0.814141
17    18  0.800253  0.828946  0.788740
18    19  0.806889  0.835005  0.795665
19    20  0.807432  0.836064  0.797086
20  MEAN  0.808454  0.835403  0.796402


In [40]:
out_path = "leiden_reconstructed_knn_cosine.csv"
df_out.to_csv(out_path, index=False)

print(f"CSV salvato in: {out_path}")

CSV salvato in: leiden_reconstructed_knn_cosine.csv


**UNICO CODICE CHE RUNNA TUTTE LE COMBINAZIONI POSSIBILI**

Per ogni test su un grafo utilizzo lo stesso seed e ci applico le 4 metodologie diverse.
Ma ogni test dei 20 per grafo avranno seed diversi

In [ ]:
import numpy as np
import pandas as pd

from cdlib import algorithms, evaluation


N_TESTS = 20

# ---- Configurazioni grafi ----
GRAPHS = {
    "Graph_1": {"N": 500, "mu": 0.1},
    "Graph_2": {"N": 1000, "mu": 0.1},
    "Graph_3": {"N": 1000, "mu": 0.3},
}

# ---- 4 combinazioni embedding/reconstruction ----
METHODS = [
    ("knn", "dot"),
    ("knn", "cosine"),
    ("threshold", "dot"),
    ("threshold", "cosine"),
]


def pairwise_f1(pred_communities, true_communities):
    def to_labels(communities):
        lab = {}
        for cid, comm in enumerate(communities):
            for n in comm:
                lab[n] = cid
        return lab

    y_pred = to_labels(pred_communities)
    y_true = to_labels(true_communities)

    nodes = sorted(set(y_true.keys()) & set(y_pred.keys()))
    n = len(nodes)
    if n < 2:
        return np.nan

    tp = fp = fn = 0
    for i in range(n):
        ni = nodes[i]
        for j in range(i + 1, n):
            nj = nodes[j]
            same_pred = (y_pred[ni] == y_pred[nj])
            same_true = (y_true[ni] == y_true[nj])

            if same_pred and same_true:
                tp += 1
            elif same_pred and not same_true:
                fp += 1
            elif not same_pred and same_true:
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0


# ============================================================
# MAIN LOOP: un CSV per grafo, MEAN per metodo
# ============================================================
for graph_name, params in GRAPHS.items():
    rows = []

    for test_id in range(1, N_TESTS + 1):

        # grafo LFR completamente random
        G, ground_truth = generate_original_graph(
            N=params["N"],
            mu=params["mu"]
        )

        # 4 combinazioni sullo STESSO grafo
        for reconstruction, metric in METHODS:
            Gr = get_reconstructed_graph(
                G,
                reconstruction=reconstruction,
                metric=metric
            )

            leiden_coms = algorithms.leiden(Gr)

            nmi = evaluation.normalized_mutual_information(leiden_coms, ground_truth).score
            ari = evaluation.adjusted_rand_index(leiden_coms, ground_truth).score
            f1  = pairwise_f1(leiden_coms.communities, ground_truth.communities)

            rows.append({
                "graph": graph_name,
                "N": params["N"],
                "mu": params["mu"],
                "method": f"{reconstruction}+{metric}",
                "test": test_id,
                "NMI": nmi,
                "F_score": f1,
                "ARI": ari,
            })

    df = pd.DataFrame(rows)

    # ---- MEAN per metodo (4 righe) ----
    mean_df = (
        df.groupby("method")[["NMI", "F_score", "ARI"]]
          .mean()
          .reset_index()
    )
    mean_df["graph"] = graph_name
    mean_df["N"] = params["N"]
    mean_df["mu"] = params["mu"]
    mean_df["test"] = "MEAN"

    mean_df = mean_df[["graph", "N", "mu", "method", "test", "NMI", "F_score", "ARI"]]

    df_out = pd.concat([df, mean_df], ignore_index=True)

    out_path = f"leiden_{graph_name.lower()}_embeddings_results.csv"
    df_out.to_csv(out_path, index=False)

    print(f"\nSalvato: {out_path}")
    print(df_out.tail(8))


TypeError: generate_original_graph() got an unexpected keyword argument 'seed'